In [17]:
from cwfc.validation import rolling_origin_splits
from cwfc.baselines import persistence_forecast
from cwfc.metrics import mae
from cwfc.metrics import rmse
from cwfc.metrics import smape
import pandas as pd
import numpy as np


In [18]:
learning_dataframe = pd.read_csv("../learning_dataframe.csv")
splits = rolling_origin_splits(learning_dataframe)
print(len(splits))
for i, split in enumerate(splits, start=1):
    print(f"Split {i}")
    print("Train end:", split.train_end)
    print("Validation start:", split.validation_start)
    print("Validation end:", split.validation_end)
    print("--------------------")

4
Split 1
Train end: 17700
Validation start: 18000
Validation end: 23700
--------------------
Split 2
Train end: 20700
Validation start: 21000
Validation end: 26700
--------------------
Split 3
Train end: 23700
Validation start: 24000
Validation end: 29700
--------------------
Split 4
Train end: 26700
Validation start: 27000
Validation end: 32700
--------------------


In [19]:

split = splits[0]
train = learning_dataframe[learning_dataframe["timestamp"] <= split.train_end]
validation = learning_dataframe[(split.validation_start <= learning_dataframe["timestamp"]) & (learning_dataframe["timestamp"] <= split.validation_end)]
print(train.shape[0])
print(validation.shape[0])

60
20


In [20]:
results = []
for i in range(len(splits)):
    print("Fold : ", (i+1))
    split = splits[i]           
    train = learning_dataframe[learning_dataframe["timestamp"] <= split.train_end]
    validation = learning_dataframe[(split.validation_start <= learning_dataframe["timestamp"]) & (learning_dataframe["timestamp"] <= split.validation_end)]    
    predictions = persistence_forecast(train, validation, "aggregate_cpu_utilization")
    print(len(predictions))
    print(predictions.head())
    print(predictions.nunique())
    MAE = mae(predictions, validation["aggregate_cpu_utilization"])
    RMSE = rmse(predictions, validation["aggregate_cpu_utilization"])
    SMAPE = smape(predictions, validation["aggregate_cpu_utilization"])
    print(MAE)
    print(RMSE)
    print(SMAPE)
    predictions_2 = persistence_forecast(train, validation, "active_vm_count")
    MAE_2 = mae(predictions_2, validation["active_vm_count"])
    RMSE_2 = rmse(predictions_2, validation["active_vm_count"])
    SMAPE_2 = smape(predictions_2, validation["active_vm_count"])
    print(MAE_2)
    print(RMSE_2)
    print(SMAPE_2)
    results.append({ "fold": i + 1, "target": "aggregate_cpu_utilization", "MAE": MAE, "RMSE": RMSE, "SMAPE": SMAPE})
    results.append({"fold": i + 1, "target": "active_vm_count", "MAE": MAE_2, "RMSE": RMSE_2, "SMAPE" : SMAPE_2})
results_dataframe = pd.DataFrame(results)
print(results_dataframe)


Fold :  1
20
60    9.839981
61    9.839981
62    9.839981
63    9.839981
64    9.839981
dtype: float64
1
0.27591586192566214
0.34176639966456723
2.865795873524437
0.0
0.0
0.0
Fold :  2
20
70    9.662172
71    9.662172
72    9.662172
73    9.662172
74    9.662172
dtype: float64
1
0.29066594104442006
0.31765196864591033
3.0634257798356566
0.0
0.0
0.0
Fold :  3
20
80    9.291072
81    9.291072
82    9.291072
83    9.291072
84    9.291072
dtype: float64
1
0.07926405796048837
0.09338204062638598
0.8528098331930946
0.0
0.0
0.0
Fold :  4
20
90    9.393673
91    9.393673
92    9.393673
93    9.393673
94    9.393673
dtype: float64
1
0.15851978590635102
0.17966941630829897
1.7058151365747238
0.0
0.0
0.0
   fold                     target       MAE      RMSE     SMAPE
0     1  aggregate_cpu_utilization  0.275916  0.341766  2.865796
1     1            active_vm_count  0.000000  0.000000  0.000000
2     2  aggregate_cpu_utilization  0.290666  0.317652  3.063426
3     2            active_vm_count  0

In [21]:
print(results_dataframe)
mean_results = results_dataframe.groupby("target")[["MAE", "RMSE", "SMAPE"]].mean()
print(mean_results)  

   fold                     target       MAE      RMSE     SMAPE
0     1  aggregate_cpu_utilization  0.275916  0.341766  2.865796
1     1            active_vm_count  0.000000  0.000000  0.000000
2     2  aggregate_cpu_utilization  0.290666  0.317652  3.063426
3     2            active_vm_count  0.000000  0.000000  0.000000
4     3  aggregate_cpu_utilization  0.079264  0.093382  0.852810
5     3            active_vm_count  0.000000  0.000000  0.000000
6     4  aggregate_cpu_utilization  0.158520  0.179669  1.705815
7     4            active_vm_count  0.000000  0.000000  0.000000
                                MAE      RMSE     SMAPE
target                                                 
active_vm_count            0.000000  0.000000  0.000000
aggregate_cpu_utilization  0.201091  0.233117  2.121962


In [22]:
train_final = pd.read_csv("../train_final.csv")
test_dataframe = pd.read_csv("../test_dataframe.csv")
predictions_persistence_final = persistence_forecast( train_final, test_dataframe, "aggregate_cpu_utilization")

In [23]:
MAE_persistence_final = mae(
    predictions_persistence_final,
    test_dataframe["aggregate_cpu_utilization"]
)

RMSE_persistence_final = rmse(
    predictions_persistence_final,
    test_dataframe["aggregate_cpu_utilization"]
)

SMAPE_persistence_final = smape(
    predictions_persistence_final,
    test_dataframe["aggregate_cpu_utilization"]
)

In [24]:
print(MAE_persistence_final)
print(RMSE_persistence_final)
print(SMAPE_persistence_final)

0.14678245915042454
0.18478539074333752
1.5850535267467123


In [25]:
np.save("../predictions/persistence_predictions.npy", predictions_persistence_final)